<a href="https://colab.research.google.com/github/mukedon/python_studies/blob/main/zx_calculus_tutorial2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZX Calculus: A Graphical Language for Quantum Circuits
## Grounded in the Ising-Trotter and Surface Code Pipeline

This notebook is the fourth part of the series. It is designed to be read alongside the three prior works:

| Notebook | What it contributes here |
|---|---|
| `cliff_trott_ising.ipynb` | Ising Hamiltonian, Trotterization, Cliffordization, Pauli twirling |
| `7_surface_code_tutorial.ipynb` | Rotated surface code, stabilizers, MWPM decoding, lattice surgery |
| `ising_surface_code_qec.ipynb` | End-to-end QEC pipeline bridging both worlds via Cliffordization |

### What ZX calculus adds

ZX calculus is a **graphical rewriting system** for quantum circuits. Instead of matrix multiplication, you manipulate diagrams using a small set of rules. It turns out that:

- Every Clifford circuit is a ZX diagram — so everything Cliffordized in the prior notebooks can be drawn and simplified graphically.
- Surface code stabilizers are **spiders** — the fundamental ZX generators.
- Pauli twirling, Cliffordization, and lattice surgery all have clean ZX-diagram interpretations.
- ZX rewriting is the backbone of modern circuit optimisation and QEC formal verification tools.

### Colour and shape convention

This notebook follows the official colour scheme from [zxcalculus.com/accessibility.html](https://zxcalculus.com/accessibility.html), chosen to be accessible under all forms of colour-vision deficiency and to print clearly in greyscale:

| Element | Colour | Hex |
|---|---|---|
| **Z spiders** | Light green | `#D8F8D8` (RGB 216, 248, 216) |
| **X spiders** | Light red | `#E8A5A5` (RGB 232, 165, 165) |
| **Hadamard boxes** | Yellow | `#FFEB3B` |

All spiders are drawn as **rounded rectangles** (not circles), following the community recommendation on zxcalculus.com. All phase labels use **black text** on light backgrounds.

### Notebook structure

1. Installation and imports
2. The two generators: Z and X spiders
3. The six rewrite rules
4. Clifford circuits as ZX diagrams
5. The Ising Trotter step in ZX
6. Surface code stabilizers as spiders
7. Syndrome extraction circuits in ZX
8. Pauli propagation and Cliffordization via ZX
9. Lattice surgery as ZX composition
10. Discussion and further reading

---
**Prerequisites:** Sections 1-4 of `7_surface_code_tutorial.ipynb` and familiarity with Clifford circuits from `ising_surface_code_qec.ipynb`.

---
## 1. Installation and Imports

```bash
pip install qiskit qiskit-aer stim pymatching matplotlib numpy networkx
```

ZX diagrams in this notebook are drawn using plain `matplotlib`. The circuits use exactly the same parameters as `ising_surface_code_qec.ipynb` so results are directly comparable.

In [1]:
!pip install qiskit qiskit_aer stim pymatching matplotlib numpy networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.1/626.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings("ignore")

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, random_clifford
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
import stim
import pymatching

# ── Official ZX colour palette (zxcalculus.com/accessibility.html) ─────────
# These are CVD-accessible, greyscale-safe, and W3C contrast-compliant.
ZX_GREEN  = '#D8F8D8'   # Z spiders — RGB(216, 248, 216)
ZX_RED    = '#E8A5A5'   # X spiders — RGB(232, 165, 165)
ZX_YELLOW = '#FFEB3B'   # Hadamard boxes
ZX_TEXT   = '#000000'   # Black text (on all spider labels)
ZX_EDGE   = '#000000'   # Black outlines
ZX_WIRE   = '#222222'   # Dark wire colour

print("All imports successful.")
print(f"  stim      : {stim.__version__}")
print(f"  pymatching: {pymatching.__version__}")
print()
print("Official ZX colour palette (zxcalculus.com/accessibility.html):")
print(f"  Z spiders  : {ZX_GREEN}   light green")
print(f"  X spiders  : {ZX_RED}   light red")
print(f"  Hadamard   : {ZX_YELLOW}   yellow")
print(f"  Node shape : rounded rectangles (community recommendation)")
print(f"  Text       : black on light background")

# ── Global parameters (identical to ising_surface_code_qec.ipynb) ──────────
N_QUBITS      = 4
J             = 0.2
H_FIELD       = 1.2
ALPHA         = np.pi / 8
TOTAL_TIME    = 1.0
P_PHYS        = 1e-2
CODE_DIST     = 3
QEC_ROUNDS    = CODE_DIST
NUM_SHOTS     = 20_000

print("\nParameters loaded (identical to ising_surface_code_qec.ipynb).")

**Reading the import block.**  
The same two ecosystems from `ising_surface_code_qec.ipynb` are loaded here: Qiskit for circuit construction and Aer simulation, and stim/pymatching for stabilizer-level QEC. The global parameters are deliberately identical.

The ZX colour constants (`ZX_GREEN`, `ZX_RED`) are defined once and used by every drawing helper in the notebook. The official palette uses **light fills with black text** — the opposite of saturated colours with white text. This is intentional: light colours print clearly in greyscale and remain distinguishable under deuteranopia and protanopia, as verified in the Jupyter notebook linked from zxcalculus.com/accessibility.html.

---
## 2. The Two Generators: Z and X Spiders

ZX calculus has exactly two generators, called **spiders**.

### 2.1 Z spider

A Z spider with $n$ input wires, $m$ output wires, and phase $\alpha \in [0, 2\pi)$:
$$Z(\alpha)_{n,m} = |0\rangle^{\otimes m}\langle 0|^{\otimes n} + e^{i\alpha}|1\rangle^{\otimes m}\langle 1|^{\otimes n}$$

Drawn as a **light green rounded rectangle**. When $\alpha = 0$, the label is omitted — this zero-phase convention is used consistently throughout.

### 2.2 X spider

An X spider with $n$ inputs, $m$ outputs, and phase $\alpha$:
$$X(\alpha)_{n,m} = |+\rangle^{\otimes m}\langle +|^{\otimes n} + e^{i\alpha}|-\rangle^{\otimes m}\langle -|^{\otimes n}$$

Drawn as a **light red rounded rectangle**. Unlabelled when $\alpha = 0$.

### 2.3 Gate dictionary

| Gate | ZX description |
|---|---|
| $Z$ | Z spider, $\alpha = \pi$ |
| $X$ | X spider, $\alpha = \pi$ |
| $S$ | Z spider, $\alpha = \pi/2$ |
| $R_z(\theta)$ | Z spider, $\alpha = \theta$ |
| $R_x(\theta)$ | X spider, $\alpha = \theta$ |
| $H$ | Yellow Hadamard box |
| CNOT | Phaseless Z spider (ctrl) wired to phaseless X spider (tgt) |

In [ ]:
# ── Shared drawing helpers ────────────────────────────────────────────────────

def spider(ax, x, y, color, label='', w=0.54, h=0.38, r=0.09, fs=9, zo=5):
    """Rounded-rectangle spider (official community shape).
    label is omitted when phase is 0, per convention.
    """
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle=f'round,pad={r}',
                          facecolor=color, edgecolor=ZX_EDGE, lw=1.6, zorder=zo)
    ax.add_patch(box)
    if label:
        ax.text(x, y, label, ha='center', va='center',
                fontsize=fs, color=ZX_TEXT, fontfamily='serif', zorder=zo+1)


def hbox(ax, x, y, w=0.40, h=0.38, r=0.07, fs=9, zo=5):
    """Yellow Hadamard box."""
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle=f'round,pad={r}',
                          facecolor=ZX_YELLOW, edgecolor=ZX_EDGE, lw=1.6, zorder=zo)
    ax.add_patch(box)
    ax.text(x, y, 'H', ha='center', va='center',
            fontsize=fs, color=ZX_TEXT, fontfamily='serif', fontweight='bold', zorder=zo+1)


def seg(ax, x0, x1, y0, y1=None, lw=1.8, color=None, ls='-'):
    if y1 is None: y1 = y0
    if color is None: color = ZX_WIRE
    ax.plot([x0, x1], [y0, y1], color=color, lw=lw, ls=ls, zorder=2)


def vseg(ax, x, y0, y1, lw=1.8):
    ax.plot([x, x], [y0, y1], color=ZX_WIRE, lw=lw, zorder=2)


def eq(ax, x, y, fs=16):
    ax.text(x, y, '=', ha='center', va='center', fontsize=fs, color='#333')


# ── Figure: generators ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(13, 2.6))
for ax in axes:
    ax.set_xlim(-1.4, 1.4); ax.set_ylim(-0.8, 0.8)
    ax.set_aspect('equal'); ax.axis('off')

# Phaseless Z spider (label omitted — zero-phase convention)
seg(axes[0], -1.2, -0.27, 0); spider(axes[0], 0, 0, ZX_GREEN, ''); seg(axes[0], 0.27, 1.2, 0)
axes[0].set_title('Z spider  (phaseless, α=0)', fontsize=9)

# Z spider with phase α
seg(axes[1], -1.2, -0.27, 0); spider(axes[1], 0, 0, ZX_GREEN, 'α'); seg(axes[1], 0.27, 1.2, 0)
axes[1].set_title('Z spider  (phase α)', fontsize=9)

# Phaseless X spider
seg(axes[2], -1.2, -0.27, 0); spider(axes[2], 0, 0, ZX_RED, ''); seg(axes[2], 0.27, 1.2, 0)
axes[2].set_title('X spider  (phaseless, α=0)', fontsize=9)

# Hadamard box
seg(axes[3], -1.2, -0.20, 0); hbox(axes[3], 0, 0); seg(axes[3], 0.20, 1.2, 0)
axes[3].set_title('Hadamard box', fontsize=9)

fig.suptitle('ZX Calculus Generators — official colours (zxcalculus.com)',
             fontsize=11, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print(f"Z colour : {ZX_GREEN}  |  X colour : {ZX_RED}")
print("Shape    : rounded rectangles  |  Text: black on light fill")
print("Phase 0  : label omitted (phaseless convention)")

**Reading the generators figure.**  
The four panels show the complete set of ZX primitives:

- *Light green rounded rectangles (Z-type)* are diagonal in the computational basis $\{|0\rangle, |1\rangle\}$. A phaseless Z spider (no label, $\alpha=0$) acts as a copy map in the Z basis. Adding phase $\alpha$ introduces a relative phase between the $|0\rangle$ and $|1\rangle$ branches.
- *Light red rounded rectangles (X-type)* are diagonal in $\{|+\rangle, |-\rangle\}$. They are copy maps in the X basis.
- *Yellow boxes (Hadamard)* convert between Z and X colour — the colour-change rule (cc).

The zero-phase convention is followed strictly: an unlabelled spider always means $\alpha = 0$, not an unknown phase. This matches the convention used in the PennyLane ZX tutorial and van de Wetering (2020). Wires run left-to-right; zero-input wires prepare states, zero-output wires perform measurements.

---
## 3. The Six Rewrite Rules

ZX calculus is **sound and complete** for Clifford+T circuits. Rule names follow van de Wetering (2020) and the PennyLane ZX tutorial. **In all rules, colours are interchangeable** — a rule shown for green spiders holds equally when all colours are swapped.

| Label | Name | What it says |
|---|---|---|
| **(f)** | **Fuse** | Same-colour spiders connected by a wire fuse; phases add |
| **(id)** | **Identity** | Phaseless 1-in 1-out spider = plain wire |
| **(π)** | **π-copy** | Phase-π spider of one colour passes through opposite colour, negating phase |
| **(c)** | **State-copy** | Basis state ($a\pi$ phase) of one colour copies through opposite-colour spider |
| **(cc)** | **Colour-change** | H-conjugation converts Z spider to X spider (same phase) |
| **(b)** | **Bialgebra** | Z-copy and X-add satisfy the bialgebra relation (CNOT structure) |

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(12, 13.5))
for ax in axes:
    ax.set_aspect('equal'); ax.axis('off')

# ── (f) Fuse ──────────────────────────────────────────────────────────────────
ax = axes[0]; ax.set_xlim(0, 12); ax.set_ylim(-0.7, 0.7)
ax.set_title('(f) Fuse — same-colour spiders connected by a wire fuse; phases add',
             fontsize=9, loc='left', color='#333')
seg(ax, 0.1, 0.73, 0); spider(ax, 1.0, 0, ZX_GREEN, 'α', w=0.54)
seg(ax, 1.27, 2.23, 0); spider(ax, 2.5, 0, ZX_GREEN, 'β', w=0.54)
seg(ax, 2.77, 3.5, 0)
eq(ax, 4.5, 0)
seg(ax, 5.4, 6.03, 0); spider(ax, 6.4, 0, ZX_GREEN, 'α+β', w=0.74)
seg(ax, 6.77, 7.5, 0)

# ── (id) Identity ─────────────────────────────────────────────────────────────
ax = axes[1]; ax.set_xlim(0, 12); ax.set_ylim(-0.7, 0.7)
ax.set_title('(id) Identity — phaseless 1-in 1-out spider is a plain wire',
             fontsize=9, loc='left', color='#333')
seg(ax, 0.1, 0.73, 0); spider(ax, 1.0, 0, ZX_GREEN, '', w=0.54)
seg(ax, 1.27, 3.5, 0)
eq(ax, 4.5, 0)
seg(ax, 5.5, 8.0, 0)

# ── (π) π-copy ────────────────────────────────────────────────────────────────
ax = axes[2]; ax.set_xlim(0, 12); ax.set_ylim(-0.7, 0.7)
ax.set_title('(π) π-copy — X(π) passes through Z(α), negating the phase (Pauli propagation)',
             fontsize=9, loc='left', color='#333')
seg(ax, 0.1, 0.73, 0); spider(ax, 1.0, 0, ZX_RED, 'π', w=0.54)
seg(ax, 1.27, 1.93, 0); spider(ax, 2.2, 0, ZX_GREEN, 'α', w=0.54)
seg(ax, 2.47, 3.5, 0)
eq(ax, 4.5, 0)
seg(ax, 5.4, 6.03, 0); spider(ax, 6.32, 0, ZX_GREEN, '−α', w=0.62)
seg(ax, 6.63, 7.33, 0); spider(ax, 7.6, 0, ZX_RED, 'π', w=0.54)
seg(ax, 7.87, 8.6, 0)

# ── (c) State-copy ────────────────────────────────────────────────────────────
ax = axes[3]; ax.set_xlim(0, 12); ax.set_ylim(-1.0, 1.0)
ax.set_title('(c) State-copy — basis state (aπ phase) copied through opposite-colour spider',
             fontsize=9, loc='left', color='#333')
seg(ax, 0.1, 0.73, 0); spider(ax, 1.0, 0, ZX_RED, 'aπ', w=0.62)
seg(ax, 1.31, 1.93, 0); spider(ax, 2.2, 0, ZX_GREEN, '', w=0.54)   # phaseless Z copy
seg(ax, 2.47, 3.1, 0, 0.55); seg(ax, 2.47, 3.1, 0, -0.55)
eq(ax, 4.5, 0)
seg(ax, 5.5, 6.13, 0, 0.55); spider(ax, 6.4, 0.55, ZX_RED, 'aπ', w=0.62)
seg(ax, 6.71, 7.5, 0.55, 0.55)
seg(ax, 5.5, 6.13, 0, -0.55); spider(ax, 6.4, -0.55, ZX_RED, 'aπ', w=0.62)
seg(ax, 6.71, 7.5, -0.55, -0.55)

# ── (cc) Colour-change ────────────────────────────────────────────────────────
ax = axes[4]; ax.set_xlim(0, 12); ax.set_ylim(-0.7, 0.7)
ax.set_title('(cc) Colour-change — H conjugation converts Z spider to X spider (same phase)',
             fontsize=9, loc='left', color='#333')
seg(ax, 0.1, 0.60, 0); hbox(ax, 0.80, 0, w=0.38)
seg(ax, 1.00, 1.63, 0); spider(ax, 1.9, 0, ZX_GREEN, 'α', w=0.54)
seg(ax, 2.17, 2.80, 0); hbox(ax, 3.00, 0, w=0.38)
seg(ax, 3.19, 3.8, 0)
eq(ax, 4.5, 0)
seg(ax, 5.3, 5.93, 0); spider(ax, 6.2, 0, ZX_RED, 'α', w=0.54)
seg(ax, 6.47, 7.2, 0)

# ── (b) Bialgebra ─────────────────────────────────────────────────────────────
ax = axes[5]; ax.set_xlim(0, 12); ax.set_ylim(-1.1, 1.1)
ax.set_title('(b) Bialgebra — Z copy and X add (encodes CNOT entanglement and stabilizer structure)',
             fontsize=9, loc='left', color='#333')
seg(ax, 0.1, 0.73, 0); spider(ax, 1.0, 0, ZX_GREEN, '', w=0.54)    # Z copy
seg(ax, 1.27, 1.88, 0, 0.6); seg(ax, 1.27, 1.88, 0, -0.6)
spider(ax, 2.15, 0.6, ZX_RED, '', w=0.54); spider(ax, 2.15, -0.6, ZX_RED, '', w=0.54)
seg(ax, 2.42, 3.2, 0.6, 0.6); seg(ax, 2.42, 3.2, -0.6, -0.6)
eq(ax, 4.5, 0)
seg(ax, 5.6, 6.23, 0.6, 0.6); seg(ax, 5.6, 6.23, -0.6, -0.6)
spider(ax, 6.5, 0.6, ZX_GREEN, '', w=0.54); spider(ax, 6.5, -0.6, ZX_GREEN, '', w=0.54)
seg(ax, 6.77, 7.5, 0.6, 0); seg(ax, 6.77, 7.5, -0.6, 0)
spider(ax, 7.77, 0, ZX_RED, '', w=0.54); seg(ax, 8.04, 8.8, 0)

fig.suptitle('The Six ZX Rewrite Rules', fontsize=13, fontweight='bold', y=1.0)
plt.tight_layout(h_pad=1.0)
plt.show()

**Reading the rewrite rules.**  
Each rule is an equation between ZX diagrams that can be applied to any sub-diagram in any order. All rules are colour-symmetric.

**(f) Fuse:** The workhorse. Adjacent same-colour spiders fuse and phases add. Two Z spiders of phases $\pi/2$ and $\pi/2$ fuse to Z($\pi$) — the $Z$ gate. This is what Qiskit's `optimization_level=1` does, stated graphically.

**(id) Identity:** A phaseless 1-in–1-out spider vanishes. Every $R_z(0)$ or $R_x(0)$ gate disappears without changing the linear map.

**(π) π-copy:** The ZX encoding of Pauli commutation. An X error before $R_z(\alpha)$ emerges on the other side as an X error with phase $-\alpha$. `apply_pauli_twirl()` in `ising_surface_code_qec.ipynb` uses exactly this rule.

**(c) State-copy:** A computational basis state (phase $0$ or $\pi$) of one colour, fed into a spider of the opposite colour, is copied to every output. This underlies the fan-out structure inside stabilizer measurements.

**(cc) Colour-change:** $HZH = X$ and $HXH = Z$ stated graphically. A Z spider with H boxes on every wire becomes an X spider of the same phase.

**(b) Bialgebra:** The deepest rule, encoding how Z-copy and X-add interact. It is needed to verify that Z errors on the control of a CNOT propagate to both wires — the key fact behind surface code stabilizer measurements.

---
## 4. Clifford Circuits as ZX Diagrams

Every gate in `STIM_GATE_MAP` from `ising_surface_code_qec.ipynb` is a ZX diagram.

### 4.1 Single-qubit Clifford gates

| Gate | ZX | Phase |
|---|---|---|
| $I$ | Z spider — vanishes by (id) | $\alpha = 0$ (unlabelled) |
| $Z$ | Z spider | $\pi$ |
| $S$ | Z spider | $\pi/2$ |
| $S^\dagger$ | Z spider | $3\pi/2$ |
| $X$ | X spider | $\pi$ |
| $\sqrt{X}$ (`sx`) | X spider | $\pi/2$ |
| $H$ | Yellow box |

### 4.2 CNOT

Phaseless Z spider on the control wire, connected by an internal vertical wire to a phaseless X spider on the target wire. Both spiders are unlabelled ($\alpha = 0$).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax in axes:
    ax.set_aspect('equal'); ax.axis('off')

# ── Panel A: CNOT ─────────────────────────────────────────────────────────────
ax = axes[0]; ax.set_xlim(-0.4, 4.0); ax.set_ylim(-1.5, 1.3)
ax.set_title('CNOT as ZX diagram', fontsize=10)
seg(ax, -0.2, 0.73, 0.7);  spider(ax, 1.0, 0.7,  ZX_GREEN, '', w=0.54)  # ctrl Z
vseg(ax, 1.0, 0.51, -0.51)
seg(ax, -0.2, 0.73, -0.7); spider(ax, 1.0, -0.7, ZX_RED,   '', w=0.54)  # tgt X
seg(ax, 1.27, 3.6, 0.7); seg(ax, 1.27, 3.6, -0.7)
ax.text(-0.25, 0.7,  'ctrl', ha='right', fontsize=8, color='#444')
ax.text(-0.25, -0.7, 'tgt',  ha='right', fontsize=8, color='#444')
ax.text(3.65,  0.7,  'ctrl', ha='left',  fontsize=8, color='#444')
ax.text(3.65,  -0.7, 'tgt',  ha='left',  fontsize=8, color='#444')

# ── Panel B: Z-error propagation via rule (b) ─────────────────────────────────
ax = axes[1]; ax.set_xlim(-0.4, 5.4); ax.set_ylim(-1.5, 1.3)
ax.set_title('Z error on control → both wires\n(bialgebra rule b)', fontsize=9)
# Z(π) before ctrl
seg(ax, -0.2, 0.23, 0.7); spider(ax, 0.5, 0.7, ZX_GREEN, 'π', w=0.52)
seg(ax, 0.76, 1.44, 0.7); spider(ax, 1.72, 0.7,  ZX_GREEN, '', w=0.54)  # CNOT ctrl
vseg(ax, 1.72, 0.51, -0.51)
seg(ax, -0.2, 1.44, -0.7); spider(ax, 1.72, -0.7, ZX_RED,   '', w=0.54)  # CNOT tgt
seg(ax, 2.00, 2.6, 0.7); seg(ax, 2.00, 2.6, -0.7)
ax.text(3.05, 0.0, '= (b)', ha='center', fontsize=8, style='italic', color='#555')
# Z errors on both wires after
seg(ax, 2.6, 3.24, 0.7); spider(ax, 3.5, 0.7,  ZX_GREEN, 'π', w=0.52)
seg(ax, 3.76, 5.0, 0.7)
seg(ax, 2.6, 3.24, -0.7); spider(ax, 3.5, -0.7, ZX_GREEN, 'π', w=0.52)
seg(ax, 3.76, 5.0, -0.7)

# ── Panel C: gate → ZX phase reference ───────────────────────────────────────
ax = axes[2]; ax.set_xlim(0, 5.5); ax.set_ylim(-0.5, 5.8)
ax.set_title('Clifford basis → ZX phases\n(unlabelled = phaseless, α=0)', fontsize=9)
ax.axis('off')
gate_rows = [
    ('I',        ZX_GREEN, '',      0.0),
    ('Z',        ZX_GREEN, 'π',     1.0),
    ('S',        ZX_GREEN, 'π/2',   2.0),
    ('S†',       ZX_GREEN, '3π/2',  3.0),
    ('X',        ZX_RED,   'π',     4.0),
    ('√X (sx)',  ZX_RED,   'π/2',   5.0),
]
for name, col, phase, ypos in gate_rows:
    seg(ax, 0.3, 0.93, ypos); spider(ax, 1.2, ypos, col, phase, w=0.52, h=0.34, fs=8)
    seg(ax, 1.47, 2.1, ypos)
    ax.text(2.2, ypos, f'← {name}', fontsize=9, va='center', color='#222')

plt.suptitle('CNOT in ZX and Error Propagation', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Reading the CNOT and error propagation figure.**

*Left (CNOT):* A phaseless Z spider (light green, unlabelled) on the control wire is connected by a vertical internal wire to a phaseless X spider (light red, unlabelled) on the target. Both are unlabelled because $\alpha = 0$ — no label is written, following the zero-phase convention from zxcalculus.com and the PennyLane tutorial.

*Centre (Z error propagation):* A Z($\pi$) spider before the CNOT control produces Z($\pi$) errors on **both** wires after, by the bialgebra rule (b). This is $\mathrm{CX}^\dagger(Z\otimes I)\mathrm{CX} = Z\otimes Z$, stated graphically. In the surface code this explains how Z-type data errors propagate into the ancilla during syndrome extraction.

*Right (gate dictionary):* Every gate in `STIM_GATE_MAP` from `ising_surface_code_qec.ipynb` maps to a spider with a specific phase. The identity gate ($I$) maps to a phaseless Z spider — which immediately vanishes by the identity rule (id), leaving only a plain wire.

---
## 5. The Ising Trotter Step in ZX

The one-step Ising Trotter circuit on two qubits:
$$\mathrm{CX}_{01} \to R_z(2J\Delta t)_1 \to \mathrm{CX}_{01} \to R_x(2h\Delta t)_0 \otimes R_x(2h\Delta t)_1$$

In ZX: each CX is a phaseless Z-X spider pair; each $R_z(\theta)$ is a Z spider labelled $\theta$; each $R_x(\theta)$ is an X spider labelled $\theta$. After **Cliffordization**, all non-Clifford phases are replaced by random multiples of $\pi/2$, making all labels Clifford-legal and the circuit stim-simulable.

In [ ]:
def build_trotter_ising(n_qubits, J, h, total_time, steps):
    qc = QuantumCircuit(n_qubits)
    dt = total_time / steps
    for _ in range(steps):
        for i in range(n_qubits - 1):
            qc.cx(i, i + 1); qc.rz(2 * J * dt, i + 1); qc.cx(i, i + 1)
        for i in range(n_qubits):
            qc.rx(2 * h * dt, i)
        qc.barrier()
    return qc

qc_2q = build_trotter_ising(2, J, H_FIELD, TOTAL_TIME, 1)
print("One Trotter step, 2 qubits — circuit diagram:")
display(qc_2q.draw('text', fold=-1))

dt = TOTAL_TIME / 1
theta_zz = 2 * J * dt
theta_x  = 2 * H_FIELD * dt
print(f"\nZX spider phases:")
print(f"  ZZ interaction : θ_ZZ = {theta_zz:.4f} rad  (non-Clifford Z spider)")
print(f"  X-field        : θ_X  = {theta_x:.4f} rad  (non-Clifford X spider)")
print(f"  Cliffordization: each → random element of {{0, π/2, π, 3π/2}}")

In [ ]:
θzz = f'{theta_zz:.2f}';  θx = f'{theta_x:.2f}'
Q0, Q1 = 0.7, -0.7

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.set_xlim(-0.4, 14.5); ax.set_ylim(-2.0, 1.8)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('ZX Diagram — One Ising Trotter Step (2 qubits)\n'
             'Left: original (non-Clifford phases).   Right: after Cliffordization.',
             fontsize=10)

# Input labels
for y, lbl in [(Q0, 'q0'), (Q1, 'q1')]:
    seg(ax, 0.0, 0.73, y); ax.text(-0.05, y, lbl, ha='right', fontsize=9, color='#555')

# CX1
spider(ax, 1.0, Q0, ZX_GREEN, '', w=0.54); vseg(ax, 1.0, Q0-0.19, Q1+0.19)
spider(ax, 1.0, Q1, ZX_RED,   '', w=0.54)
seg(ax, 1.27, 2.4, Q0); seg(ax, 1.27, 2.4, Q1)
ax.text(1.0, 1.25, 'CX₁', fontsize=8, ha='center', color='#555')

# Rz on q1 (non-Clifford)
spider(ax, 2.7, Q1, ZX_GREEN, θzz, w=0.66)
seg(ax, 2.4, 2.4, Q0); seg(ax, 3.03, 4.1, Q1); seg(ax, 2.4, 4.1, Q0)
ax.text(2.7, Q1-0.65, f'Rz({θzz})\nnon-Clifford', fontsize=7, ha='center', color='#5a5a00')

# CX2
spider(ax, 4.4, Q0, ZX_GREEN, '', w=0.54); vseg(ax, 4.4, Q0-0.19, Q1+0.19)
spider(ax, 4.4, Q1, ZX_RED,   '', w=0.54)
seg(ax, 4.67, 5.6, Q0); seg(ax, 4.67, 5.6, Q1)
ax.text(4.4, 1.25, 'CX₂', fontsize=8, ha='center', color='#555')

# Rx on each qubit (non-Clifford)
spider(ax, 5.9, Q0, ZX_RED, θx, w=0.66); spider(ax, 5.9, Q1, ZX_RED, θx, w=0.66)
seg(ax, 5.6, 5.6, Q0); seg(ax, 5.6, 5.6, Q1)
seg(ax, 6.23, 7.2, Q0); seg(ax, 6.23, 7.2, Q1)
ax.text(5.9, 1.25, f'Rx({θx})\nnon-Clifford', fontsize=7, ha='center', color='#5a0000')

# Arrow
ax.annotate('', xy=(8.5, 0.0), xytext=(7.5, 0.0),
            arrowprops=dict(arrowstyle='->', color='#444', lw=2.2))
ax.text(8.0, 0.42, 'Cliffordize\n(f + id)', ha='center', fontsize=8, color='#333')

# Cliffordized version
seg(ax, 8.6, 9.23, Q0); seg(ax, 8.6, 9.23, Q1)
spider(ax, 9.5, Q0, ZX_GREEN, '', w=0.54); vseg(ax, 9.5, Q0-0.19, Q1+0.19)
spider(ax, 9.5, Q1, ZX_RED,   '', w=0.54)
seg(ax, 9.77, 10.5, Q0); seg(ax, 9.77, 10.5, Q1)

# Clifford Rz (π/2 example)
spider(ax, 10.8, Q1, ZX_GREEN, 'π/2', w=0.68)
seg(ax, 10.5, 10.5, Q0); seg(ax, 11.14, 12.1, Q1); seg(ax, 10.5, 12.1, Q0)
ax.text(10.8, Q1-0.65, 'S gate\n(Clifford)', fontsize=7, ha='center', color='#004000')

spider(ax, 12.4, Q0, ZX_GREEN, '', w=0.54); vseg(ax, 12.4, Q0-0.19, Q1+0.19)
spider(ax, 12.4, Q1, ZX_RED,   '', w=0.54)
seg(ax, 12.67, 13.4, Q0); seg(ax, 12.67, 13.4, Q1)

# Clifford Rx (π on q0; identity on q1 — unlabelled, omitted by convention)
spider(ax, 13.7, Q0, ZX_RED, 'π',  w=0.54)   # X gate
# q1 Rx → identity → omitted by (id), so just a wire
seg(ax, 13.4, 14.3, Q1)
ax.text(13.7, 1.25, 'Clifford phases\nall ∈{0,π/2,π,3π/2}',
        fontsize=7.5, ha='center', color='#534AB7')

plt.tight_layout()
plt.show()

**Reading the Ising Trotter ZX diagram.**  
The left half is the original diagram. Z spiders labelled $0.40$ and X spiders labelled $2.40$ are *non-Clifford* — their phases are not multiples of $\pi/2$ — which prevents stim simulation.

The right half shows the Cliffordized version. Three changes are visible:
1. The non-Clifford Z($\theta_{ZZ}$) spider between the two CX gates becomes Z($\pi/2$) — an $S$ gate — a Clifford phase.
2. The $R_x$ on q0 becomes X($\pi$) — an $X$ gate.
3. The $R_x$ on q1 becomes X($0$) — a phaseless identity spider — which **vanishes entirely** by rule (id). No rounded rectangle is drawn for it, consistent with the zero-phase-omission convention.

The vertical internal wires (CX connectivity) are **identical** in both halves. The Merkel et al. PTA theorem, in ZX language: *any Clifford diagram with the same spider connectivity has the same logical error rate under QEC.* Cliffordization preserves connectivity; only phase labels change.

---
## 6. Surface Code Stabilizers as Spiders

The $d=3$ rotated surface code stabilizers from `7_surface_code_tutorial.ipynb` are ZX spiders:

- **Z-type plaquette stabilizer** $Z^{\otimes 4}$: a Z spider (light green rounded rectangle) at the tile centre with 4 data-qubit legs.
- **X-type vertex stabilizer** $X^{\otimes 4}$: an X spider (light red rounded rectangle) with 4 data-qubit legs.
- **Commutativity:** Adjacent Z–X pairs share 0 or 2 data qubits (even overlap). By the bialgebra rule (b), even-overlap pairs commute — the ZX proof that all stabilizers mutually commute.

The full ancilla-mediated measurement circuit — Z-cap → four CX Z-X pairs → Z-cup — collapses to a **single Z spider with 4 data-qubit wires** by the fuse rule (f). The measurement circuit and the stabilizer operator are the same ZX object.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

# ── Left: d=3 layout with ZX spiders ─────────────────────────────────────────
ax = axes[0]; ax.set_xlim(-0.9, 2.9); ax.set_ylim(-0.9, 2.85)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('d=3 Rotated Surface Code\nStabilizers as ZX Spiders', fontsize=10)

d = 3
tcol = {'X': ZX_RED, 'Z': ZX_GREEN}
for row in range(d-1):
    for col in range(d-1):
        ptype = 'X' if (row+col)%2 == 0 else 'Z'
        ax.add_patch(plt.Polygon(
            [(col,row),(col+1,row),(col+1,row+1),(col,row+1)],
            closed=True, color=tcol[ptype], alpha=0.17, zorder=0))
        cx_c, cy_c = col+0.5, row+0.5
        for dr,dc in [(-0.47,-0.47),(-0.47,0.47),(0.47,-0.47),(0.47,0.47)]:
            qr,qc = cy_c+dr, cx_c+dc
            if 0<=qr<=d-1 and 0<=qc<=d-1:
                ax.plot([cx_c,qc],[cy_c,qr], color='#777', lw=1.1, alpha=0.5, zorder=3)
        spider(ax, cx_c, cy_c, tcol[ptype], ptype, w=0.44, h=0.32, fs=8, zo=6)

# Data qubits as small white rounded rectangles
for r in range(d):
    for c in range(d):
        ax.add_patch(FancyBboxPatch((c-0.14, r-0.14), 0.28, 0.28,
                                    boxstyle='round,pad=0.03',
                                    facecolor='white', edgecolor='#333', lw=1.5, zorder=7))
        ax.text(c, r, 'D', ha='center', va='center', fontsize=7, color='#333', zorder=8)

ax.legend(handles=[
    mpatches.Patch(facecolor=ZX_RED,   edgecolor='#333', label='X spider (X stabilizer)'),
    mpatches.Patch(facecolor=ZX_GREEN, edgecolor='#333', label='Z spider (Z stabilizer)'),
    mpatches.Patch(facecolor='white',  edgecolor='#333', label='Data qubit'),
], loc='lower left', bbox_to_anchor=(0,-0.32), ncol=1, fontsize=8)

# ── Right: Z stabilizer measurement gadget ────────────────────────────────────
ax = axes[1]; ax.set_xlim(-0.5, 9.5); ax.set_ylim(-1.9, 3.8)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Z Stabilizer Measurement as ZX\nBefore (top) and after fuse rule (f) (bottom)', fontsize=10)

data_ys = [3.2, 2.2, 1.2, 0.2]
anc_y   = -0.6

# Ancilla |0> preparation
spider(ax, 0.5, anc_y, ZX_GREEN, '|0⟩', w=0.72, h=0.36, fs=8)
prev_x = 0.86

cx_xs = [1.8, 2.9, 4.0, 5.1]
for i, (dy, cx_x) in enumerate(zip(data_ys, cx_xs)):
    # Data wire
    seg(ax, -0.3, cx_x-0.27, dy)
    spider(ax, cx_x, dy, ZX_GREEN, '', w=0.50, h=0.32, fs=8)  # Z ctrl, phaseless
    seg(ax, cx_x+0.25, 7.5, dy)
    vseg(ax, cx_x, dy-0.16, anc_y+0.18)
    # Ancilla wire
    seg(ax, prev_x, cx_x-0.25, anc_y)
    spider(ax, cx_x, anc_y, ZX_RED, '', w=0.50, h=0.32)        # X ancilla
    prev_x = cx_x + 0.25

# Measurement
seg(ax, prev_x, 5.9, anc_y)
spider(ax, 6.2, anc_y, ZX_GREEN, 'M', w=0.54, h=0.36, fs=8)
ax.annotate('', xy=(6.7, anc_y), xytext=(6.47, anc_y),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
ax.text(6.85, anc_y, 's (syndrome bit)', fontsize=7.5, va='center', color='#555')

# Fusion arrow
ax.annotate('', xy=(4.0,-1.2), xytext=(4.0,-0.88),
            arrowprops=dict(arrowstyle='->', color='#444', lw=2.0))
ax.text(4.4,-1.04, 'fuse (f)', fontsize=8, va='center', color='#333', style='italic')

# Fused single Z spider
fused_x, fused_y = 4.0, -1.65
spider(ax, fused_x, fused_y, ZX_GREEN, 'Z', w=0.68, h=0.42, fs=11, zo=5)
for dy in data_ys:
    ax.plot([fused_x, 7.5], [fused_y+0.21, dy],
            color=ZX_WIRE, lw=1.2, alpha=0.45, ls='--', zorder=2)
ax.text(7.7, fused_y, '→ s', fontsize=9, va='center', color='#333', style='italic')
ax.text(0.5, fused_y, 'One Z spider\n4 data-qubit legs', fontsize=8, ha='center',
        color='#333', va='center')

plt.suptitle('Surface Code Stabilizers as ZX Spiders', fontsize=12, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()

**Reading the stabilizer figures.**

*Left panel:* The $d=3$ code from `7_surface_code_tutorial.ipynb` is redrawn with ZX spiders at tile centres — light green for Z-type, light red for X-type. All nodes are rounded rectangles with black text. The checkerboard alternation guarantees even Z–X overlap, giving commutativity via the bialgebra rule (b).

*Right panel:* The top half shows the explicit Z stabilizer measurement circuit as a ZX diagram: a Z spider cap ($|0\rangle$), four phaseless CX Z-X pairs with vertical internal wires, and a Z spider cup (measurement $M$). The fuse rule (f) collapses all Z-type nodes (cap + four CX controls + cup) into one big Z spider — the bottom diagram. The resulting single light green rounded rectangle with 4 data-qubit legs *is* the $Z^{\otimes 4}$ stabilizer. The fusion makes visually immediate what `make_memory_circuit()` achieves algorithmically.

---
## 7. Syndrome Extraction Circuits in ZX

Every `stim` instruction in the surface code memory circuit maps to a ZX primitive. The mapping is one-to-one and uses only the official light-colour rounded-rectangle notation throughout.

In [ ]:
d = CODE_DIST
circ = stim.Circuit.generated(
    "surface_code:rotated_memory_z", rounds=d, distance=d,
    after_clifford_depolarization=P_PHYS,
    before_round_data_depolarization=0.0,
    after_reset_flip_probability=0.0,
    before_measure_flip_probability=0.0,
)

print(f"d={d} surface code memory circuit — ZX statistics:")
print(f"  Qubits      : {circ.num_qubits}  = 2d²-1 = {2*d**2-1}")
print(f"  Ancillas    : {d**2-1}  = d²-1")
print(f"  Data qubits : {d**2}  = d²")
print()

gate_counts = {}
for inst in circ.flattened():
    gate_counts[inst.name] = gate_counts.get(inst.name, 0) + 1

zx_map = {
    'CX':   'Z-X spider pair (entanglement)               rule (b)',
    'H':    'Hadamard box  (colour-change)                rule (cc)',
    'R':    'Z spider cap  — ancilla |0⟩ preparation     ',
    'MR':   'Z spider cup+cap — measure then reset        ',
    'M':    'Z spider cup  — final data measurement       ',
    'TICK': 'time-layer separator                         ',
}

print("Instruction → ZX primitive:")
for k, v in sorted(gate_counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<10}: {v:>5}   {zx_map.get(k, '')}")

print()
print("Per-ancilla per-round gadget — fuse (f) reduction:")
print("  R  → Z spider cap")
print("  H  → Hadamard box (X-type ancillas only)")
print("  CX → phaseless Z-X spider pair  × 4")
print("  H  → Hadamard box (undo colour change)")
print("  MR → Z spider cup + Z spider cap")
print("  After (f): all Z nodes on ancilla wire fuse →")
print("             single Z spider with 4 data-qubit legs")

In [ ]:
# Verify ZX fusion with a minimal 2-round stabilizer circuit
mini = stim.Circuit("""
    R 4
    TICK
    CX 0 4
    TICK
    CX 1 4
    TICK
    CX 2 4
    TICK
    CX 3 4
    TICK
    MR 4
    DETECTOR rec[-1]
    TICK
    CX 0 4
    TICK
    CX 1 4
    TICK
    CX 2 4
    TICK
    CX 3 4
    TICK
    MR 4
    DETECTOR rec[-1] rec[-2]
    M 0 1 2 3
    OBSERVABLE_INCLUDE(0) rec[-1] rec[-2] rec[-3] rec[-4]
""")
print(f"Minimal 2-round Z-stabilizer: {mini.num_qubits} qubits, "
      f"{mini.num_measurements} measurements, {mini.num_detectors} detectors")
print()
print("Fuse (f) reduction — step by step:")
print("  R 4        : Z spider cap  (0 inputs, 1 output, phaseless)")
print("  CX i 4 × 4 : four Z-X spider pairs on ancilla wire")
print("  MR 4       : Z spider cup + cap")
print("  Rule (f)   : Z-cap ∘ Z-X ∘ Z-X ∘ Z-X ∘ Z-X ∘ Z-cup")
print("             = single Z spider, 4 data-qubit wires + 1 classical output")
print("  DETECTOR   : XOR of two Z-cup outputs")
print("             = fires iff Z-stabilizer changed between rounds (error!)")

**Reading the circuit statistics and fusion verification.**  
Each `stim` instruction is a ZX primitive:
- `R` is a Z spider cap (phaseless, no inputs, 1 output). `MR` is a Z spider cup (1 input, 0 outputs) immediately followed by a Z spider cap — both light green rounded rectangles.
- `H` is a Hadamard box (yellow). It appears in pairs around X-type ancilla measurements — one before (converting X basis to Z basis for measurement) and one after (restoring the measurement basis).
- `CX` is a phaseless Z-X spider pair — light green control, light red target, vertical internal wire.
- `TICK` is a pure structural separator — no ZX node, just a time-layer boundary.

The fuse reduction printed below shows, step by step, how all Z-type nodes on the ancilla wire collapse by rule (f) into one Z spider. This is the ZX proof that `make_memory_circuit()` from `ising_surface_code_qec.ipynb` correctly implements the $Z^{\otimes 4}$ stabilizer measurement.

---
## 8. Pauli Propagation and Cliffordization via ZX

### 8.1 Pauli twirling as ZX — rule (π)

`apply_pauli_twirl()` wraps every CX in random Paulis. A Pauli gate is a phase-$\pi$ spider. The π-copy rule says:
- $X(\pi) \circ Z(\alpha) = Z(-\alpha) \circ X(\pi)$ — X error conjugates Z phase, negating it.
- $Z(\pi) \circ X(\alpha) = X(-\alpha) \circ Z(\pi)$ — Z error conjugates X phase.

Averaging over random Pauli wrappings randomises all relative phases, converting coherent rotation errors into a stochastic Pauli channel — the ZX derivation of why twirling satisfies the PTA precondition.

### 8.2 Cliffordization as ZX — rules (f) and (id)

Cliffordization replaces each non-Clifford Z($\alpha$) spider with Z($k\pi/2$). The ZX topology — all wires, all spider connections — is unchanged. Since all six ZX rules operate on phase labels without modifying connectivity, **any simplification valid on the Clifford proxy applies equally to the original Ising circuit** — the ZX statement of the PTA theorem.

In [ ]:
def rx_mat(t): c,s=np.cos(t/2),np.sin(t/2); return np.array([[c,-1j*s],[-1j*s,c]])
def rz_mat(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]])
Xg = np.array([[0,1],[1,0]])
Zg = np.array([[1,0],[0,-1]])

print("π-copy rule (π) — numerical verification")
print(f"  {'α':>12}  {'X·Rz(α)=Rz(−α)·X':>22}  {'Z·Rx(α)=Rx(−α)·Z':>22}")
print("  " + "─" * 58)
for alpha in [np.pi/4, np.pi/3, 0.7, np.pi/2]:
    mz = np.allclose(Xg @ rz_mat(alpha), rz_mat(-alpha) @ Xg)
    mx = np.allclose(Zg @ rx_mat(alpha), rx_mat(-alpha) @ Zg)
    print(f"  {alpha:>12.4f}  {'✓' if mz else '✗':>22}  {'✓' if mx else '✗':>22}")

print()
print("Cliffordization closure — −α mod 2π stays Clifford:")
print(f"  {'Phase':>8}  {'−α mod 2π':>12}  {'Clifford?':>12}")
print("  " + "─" * 36)
names = {0:'0', np.pi/2:'π/2', np.pi:'π', 3*np.pi/2:'3π/2'}
for a in [0, np.pi/2, np.pi, 3*np.pi/2]:
    neg = (-a) % (2*np.pi)
    ok  = any(np.isclose(neg, k*np.pi/2) for k in range(4))
    neg_name = names.get(neg, f'{neg:.3f}')
    print(f"  {names[a]:>8}  {neg_name:>12}  {'✓ yes' if ok else '✗ no':>12}")

print()
print("Non-Clifford angle (true Ising circuit):")
nc  = np.pi/4; neg_nc = (-nc) % (2*np.pi)
ok  = any(np.isclose(neg_nc, k*np.pi/2) for k in range(4))
print(f"  −(π/4) mod 2π = {neg_nc:.4f} → Clifford: {'✓' if ok else '✗  (exits Clifford subtheory!)'}")

**Reading the Pauli propagation verification.**  
Each row confirms rule (π) numerically: $X \cdot R_z(\alpha) = R_z(-\alpha) \cdot X$ and $Z \cdot R_x(\alpha) = R_x(-\alpha) \cdot Z$ are exact matrix equalities for all tested angles.

The closure table confirms that negating any Clifford phase (multiple of $\pi/2$) yields another Clifford phase — so Pauli twirling applied to a Cliffordized circuit keeps all spider labels within the Clifford subtheory. The proxy circuits from `ising_surface_code_qec.ipynb` remain stim-simulable after arbitrary Pauli conjugation.

The final line demonstrates why the *real* Ising circuit exits the Clifford subtheory under rule (π): $-(\pi/4) \bmod 2\pi = 7\pi/4$ is not a multiple of $\pi/2$. This is the ZX proof of why `stim` cannot simulate the original Ising circuit directly — and why the Clifford proxy is essential.

---
## 9. Lattice Surgery as ZX Composition

The logical CX from `ising_surface_code_qec.ipynb` uses lattice surgery: merge (measuring $\bar{X}_c \otimes \bar{X}_t$) then split. In ZX, this is the fuse rule (f) applied to two $\bar{X}$ logical spiders across a shared boundary, followed by its reverse.

- **$\bar{X}$ logical operator:** a row of X spiders (light red) across the patch — by rule (f), this is one X spider with $d$ data-qubit wires.
- **Merge = rule (f) forward:** the two $\bar{X}$ strings fuse into a joint X spider spanning both patches.
- **Split = rule (f) reverse:** the joint spider is written as two spiders connected by an internal wire.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax in axes:
    ax.set_aspect('equal'); ax.axis('off')


def draw_patch(ax, xoff, yoff, d=3, show_X=True, label=''):
    for r in range(d):
        for c in range(d):
            ax.add_patch(FancyBboxPatch((xoff+c*0.5-0.12, yoff+r*0.5-0.12), 0.24, 0.24,
                                        boxstyle='round,pad=0.03',
                                        facecolor='white', edgecolor='#444', lw=1.4, zorder=5))
    if show_X:
        mid = d//2
        for c in range(d):
            ax.add_patch(FancyBboxPatch((xoff+c*0.5-0.13, yoff+mid*0.5-0.13), 0.26, 0.26,
                                        boxstyle='round,pad=0.03',
                                        facecolor=ZX_RED, edgecolor='#333', lw=1.3, alpha=0.85, zorder=6))
        ax.text(xoff+(d-1)*0.5+0.28, yoff+mid*0.5, r'$\bar{X}$',
                ha='left', fontsize=9, color='#800000')
    ax.add_patch(plt.Polygon(
        [(xoff-0.18,yoff-0.18),(xoff+(d-1)*0.5+0.18,yoff-0.18),
         (xoff+(d-1)*0.5+0.18,yoff+(d-1)*0.5+0.18),(xoff-0.18,yoff+(d-1)*0.5+0.18)],
        fill=False, ec='#999', lw=1.1, ls='--', zorder=1))
    ax.text(xoff+(d-1)*0.5/2, yoff-0.42, label, ha='center', fontsize=9, color='#333')


# ── Panel 1: Before merge ─────────────────────────────────────────────────────
ax = axes[0]; ax.set_xlim(-0.4, 4.6); ax.set_ylim(-0.8, 1.9)
ax.set_title('Before merge:\ntwo separate patches', fontsize=9)
draw_patch(ax, 0.0, 0.0, label='Control (c)')
draw_patch(ax, 2.4, 0.0, label='Target (t)')
ax.text(2.1, 0.5, '|', ha='center', fontsize=20, color='#ccc')

# ── Panel 2: Merge ────────────────────────────────────────────────────────────
ax = axes[1]; ax.set_xlim(-0.4, 4.6); ax.set_ylim(-0.8, 1.9)
ax.set_title('Merge phase (d rounds):\nrule (f) applied at boundary', fontsize=9)
draw_patch(ax, 0.0, 0.0, label='Control (c)')
draw_patch(ax, 2.4, 0.0, label='Target (t)')
ax.add_patch(plt.Polygon(
    [(1.92,-0.18),(2.58,-0.18),(2.58,1.18),(1.92,1.18)],
    facecolor=ZX_YELLOW, alpha=0.45, edgecolor='#c8a000', lw=1.6, zorder=0))
spider(ax, 2.25, 0.5, ZX_RED, 'X̄cX̄t', w=0.76, h=0.40, fs=7, zo=8)
for dy in [0.0, 0.5, 1.0]:
    ax.plot([1.35,1.90],[dy,0.5], color=ZX_WIRE, lw=0.9, alpha=0.35, ls='--')
    ax.plot([2.60,2.88],[0.5,dy], color=ZX_WIRE, lw=0.9, alpha=0.35, ls='--')
ax.text(2.25,-0.60, 'boundary ancillas', ha='center', fontsize=7, color='#8B7000')

# ── Panel 3: ZX diagram of logical CX ────────────────────────────────────────
ax = axes[2]; ax.set_xlim(-0.5, 6.0); ax.set_ylim(-1.2, 2.2)
ax.set_title('ZX diagram of logical CX\nmerge = (f),  split = (f) reversed', fontsize=9)

seg(ax, -0.3, 0.64, 1.2); spider(ax, 0.9, 1.2, ZX_GREEN, 'Z̄', w=0.52, h=0.36, fs=9)
seg(ax, 1.16, 5.2, 1.2)
seg(ax, -0.3, 0.64, 0.0); spider(ax, 0.9, 0.0, ZX_RED, 'X̄', w=0.52, h=0.36, fs=9)
seg(ax, 1.16, 5.2, 0.0)

# Joint XX spider
spider(ax, 2.8, 0.6, ZX_RED, 'X̄cX̄t', w=0.80, h=0.42, fs=8, zo=7)
vseg(ax, 2.8, 0.81, 1.2-0.18); vseg(ax, 2.8, 0.39, 0.0+0.18)
seg(ax, 3.2, 4.2, 0.6, 0.6, ls='--', color='#999')
ax.text(4.3, 0.6, '→ s', fontsize=9, va='center', color='#333', style='italic')

ax.text(1.95, 1.05, 'merge\n(f)', fontsize=7.5, ha='center', color='#800000')
ax.text(3.60, 1.05, 'split\n(f⁻¹)', fontsize=7.5, ha='center', color='#003070')
ax.text(-0.35, 1.2, 'ctrl', ha='right', fontsize=8, color='#444')
ax.text(-0.35, 0.0, 'tgt',  ha='right', fontsize=8, color='#444')

plt.suptitle('Lattice Surgery as ZX Spider Fusion — Rule (f)', fontsize=12,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Reading the lattice surgery ZX figure.**

*Left panel:* Two separate surface code patches, each with its own $\bar{X}$ logical string (light red rounded rectangles along the middle row). The vertical bar indicates a physical boundary — the patches are not yet connected.

*Centre panel:* During the $d$ merge rounds, ancilla qubits at the shared boundary (yellow region) measure the joint $\bar{X}_c \otimes \bar{X}_t$ stabilizer. In ZX, the two $\bar{X}$ strings fuse by rule (f) into one joint X spider (labelled X̄cX̄t) with legs into both patches. The yellow background matches the Hadamard-box colour convention — highlighting that the boundary measurement is a transversal layer in the ZX diagram.

*Right panel:* The full logical CX: control wire (Z̄, light green) and target wire (X̄, light red), with the joint XX spider between them. The dashed wire carrying $s$ is the classical measurement outcome. Merge is rule (f) forward; split is rule (f) in reverse — conceptually one line of ZX algebra, even though hardware requires $2d$ QEC rounds.

---
## 10. Discussion and Conclusions

### 10.1 ZX semantic map of the pipeline

| Pipeline step (prior notebooks) | ZX foundation | Rules |
|---|---|---|
| Trotter circuit | Z/X spiders with continuous phases | — |
| Cliffordization | Restrict phases to $k\pi/2$ | (f), (id) |
| PTA theorem | Phase-invariance of spider connectivity | (f), (id) |
| `STIM_GATE_MAP` translation | Spider phase lookup | — |
| Stabilizer generators | Multi-legged spiders | (f) |
| Syndrome extraction gadget | Cap → Z-X chain → cup | (f), (cc) |
| Pauli twirling | π-copy: errors propagate past gates | (π) |
| Lattice surgery | Fuse and split logical $\bar{X}$ strings | (f) |

### 10.2 Recommended next steps

1. **Install `pyzx`** and apply `pyzx.full_reduce()` to the 1-step Clifford proxy from `ising_surface_code_qec.ipynb`; compare gate counts before and after.
2. **T gates:** Replace Z($\pi/4$) spiders with the T-gate magic state gadget — the first step toward universal fault-tolerant computation.
3. **Decoder verification:** Draw the MWPM matching graph as a ZX diagram; verify that matched correction operators cancel detected errors using rules (π) and (b).
4. **Scale up:** Repeat for $N=8$ or $N=16$ Ising qubits and observe which fusions `full_reduce()` finds.

---

### References and colour attribution

- **Official ZX colours and shape recommendation:** [zxcalculus.com/accessibility.html](https://zxcalculus.com/accessibility.html)
- **Rule names and conventions:** van de Wetering (2020), *ZX-Calculus for the Working Quantum Computer Scientist* ([arXiv:2012.13966](https://arxiv.org/abs/2012.13966))
- **PennyLane ZX tutorial:** [pennylane.ai/qml/demos/tutorial_zx_calculus](https://pennylane.ai/qml/demos/tutorial_zx_calculus)
- **`pyzx` library:** Kissinger & van de Wetering (2019), [arXiv:1904.04735](https://arxiv.org/abs/1904.04735)
- **East et al. (2021):** Scalar conventions followed here
- **Coecke & Duncan (2008, 2011):** Original ZX calculus
- **Jeandel, Perdrix, Vilmart (2018):** Completeness of ZX for Clifford+T
- **Merkel et al. (2025):** Cliffordization and PTA theorem
- **Gidney (2021):** `stim` stabilizer simulator
- **Fowler et al. (2012):** Surface code review